# 2023年Java大厂面试｜09：JVM虚拟机篇

> 学习目标：从 JVM 运行时数据区、对象分配、类加载到垃圾回收与故障排查，形成一套能解释原理、能定位问题、能结合项目回答的面试笔记。

**资料范围**：主要依据网盘 `5.2023版Java面试教程/09-JVM虚拟机篇/PPT/09-JVM相关面试题.pptx` 在线预览，共 135 页。课件还包含“代码”“视频”“资料”和在线讲义文稿；本篇以可读的 PPT 页面为主，并补充 JDK 版本边界。


## 目录与答题主线

| 模块 | 必会问题 | 记忆主线 |
|---|---|---|
| JVM 基础 | JVM 是什么、为什么跨平台 | 字节码 + 运行时环境 |
| 运行时数据区 | 程序计数器、栈、堆、方法区、直接内存 | 线程私有/共享、存什么、何时溢出 |
| 对象与内存 | 对象创建、布局、引用类型、String 常量池 | 分配、定位、回收 |
| 类加载 | 加载、链接、初始化、双亲委派 | 类从字节码到可用 |
| 垃圾回收 | 判断存活、算法、分代、收集器 | 降低停顿与碎片 |
| 调优排障 | OOM、StackOverflow、GC 日志、JVM 参数 | 先证据后调参 |


## 一、JVM 是什么

JVM（Java Virtual Machine）是运行 Java 字节码的虚拟机，为字节码提供统一的执行环境。编译器把 `.java` 编译成与平台无关的 `.class`，不同操作系统分别提供对应的 JVM 实现，因此可以做到“一次编写、到处运行”。

JVM 的价值不仅是跨平台，还包括自动内存管理、垃圾回收、运行时动态链接和 JIT（即时编译）优化。面试中要区分：Java 语言规范定义行为，JVM 规范定义运行时抽象，HotSpot 是一种具体实现。


## 二、运行时数据区总览

JVM 运行时数据区可以按线程归属划分：

- **线程私有**：程序计数器、虚拟机栈、本地方法栈；线程结束后通常随之回收。
- **线程共享**：堆、方法区（JDK 8 的实现通常是元空间）；多个线程共同访问，需要考虑并发安全。
- **堆外区域**：直接内存不属于 JVM 堆，但会影响进程的总内存和 NIO 使用。

不同区域的典型故障不同：堆不足常见 `OutOfMemoryError: Java heap space`；栈帧过深常见 `StackOverflowError`；线程创建过多可能出现无法创建本地线程；元空间不足常见 `OutOfMemoryError: Metaspace`。


## 三、程序计数器

程序计数器（PC Register）是线程私有的一小块内存，保存当前线程下一条要执行的字节码指令地址（或行号）。线程切换后，JVM 依靠每个线程自己的计数器恢复执行位置。

执行 Native 方法时，规范没有要求记录 Java 字节码地址，因此该区域是 JVM 规范中唯一没有规定 `OutOfMemoryError` 的运行时区域。排查字节码和局部变量时可使用 `javap -v Example.class` 查看反汇编信息。


## 四、虚拟机栈与栈帧

每个线程创建时都会有一个虚拟机栈；线程每调用一个 Java 方法，就压入一个栈帧，方法返回时弹出。栈帧主要包含：

- **局部变量表**：方法参数和局部变量，`long/double` 通常占两个槽位；
- **操作数栈**：字节码指令计算的临时数据；
- **动态链接**：指向运行时常量池中方法的符号引用；
- **方法返回地址**：方法正常或异常退出后的继续位置。

递归没有终止条件、调用链过深或单个线程栈空间过小，会触发 `StackOverflowError`。栈大小可通过 `-Xss` 调整，但增大栈会减少同一进程可创建的线程数量。


## 五、本地方法栈与直接内存

本地方法栈服务于 Native 方法，具体实现由 JVM 决定，HotSpot 中通常与虚拟机栈合并管理。它与虚拟机栈的区别在于：前者主要支撑 C/C++ 等本地方法调用，后者支撑 Java 方法。

直接内存由操作系统分配，不在 Java 堆中。`ByteBuffer.allocateDirect()`、Netty 等高性能网络框架会使用它，减少 Java 堆与内核缓冲区之间的复制。直接内存不足也会导致 `OutOfMemoryError`，排查时不能只看堆大小，还要检查 `-XX:MaxDirectMemorySize`、Native Memory Tracking 和容器内存限制。


## 六、Java 堆与分代结构

堆是线程共享区域，主要存放对象实例和数组，也是垃圾回收器重点管理的区域。传统分代模型把堆划分为年轻代和老年代：年轻代包括 Eden 与两个 Survivor 区；对象首次分配通常进入 Eden，经历 Minor/Young GC 后仍存活的对象在 Survivor 间复制，达到年龄阈值或因空间担保失败时晋升老年代。

老年代主要保存存活时间较长的对象。大对象可能直接进入老年代，具体阈值取决于收集器和参数。堆无法继续扩展且没有足够空间分配对象时，会抛出 `OutOfMemoryError: Java heap space`。


### Java 7 与 Java 8 内存结构的版本差异

课件对比了 Java 7 的“方法区/永久代”和 Java 8 的“元空间”：

- Java 7 及更早的 HotSpot 常把类元数据放在永久代（PermGen）；
- Java 8 移除永久代，类元数据转移到本地内存中的 Metaspace，字符串常量池也已移到堆；
- `Method Area` 是规范概念，PermGen、Metaspace 是 HotSpot 的实现方式，不要把三者当成同一个固定区域。

常见参数：`-Xms` 初始堆、`-Xmx` 最大堆、`-Xmn` 年轻代大小（使用时要结合收集器）、`-XX:MetaspaceSize` 初始元空间阈值、`-XX:MaxMetaspaceSize` 最大元空间。


## 七、对象创建与内存布局

执行 `new` 时，JVM 先检查类是否已加载；随后在堆中分配内存、将对象字段初始化为零值、执行对象头设置和构造方法。分配方式取决于堆是否规整：指针碰撞适合规整空间，空闲列表适合碎片较多的空间。并发分配时还要通过 TLAB（Thread Local Allocation Buffer）减少线程竞争。

对象通常由三部分组成：对象头、实例数据、对齐填充。对象头包含 Mark Word、类型指针等信息；实例数据按字段存放；对象大小常需按 8 字节对齐。具体布局与压缩指针、JVM 实现及对象字段类型有关。


## 八、对象可达性与引用类型

GC 不再简单使用引用计数，而是从 GC Roots 出发做可达性分析。常见 GC Roots 包括：栈帧中的局部变量、静态字段、常量引用、JNI 引用和正在运行的线程等。无法从 Roots 到达的对象才具备被回收的资格，但最终回收前可能经历 `finalize`（已不推荐使用，现代 Java 应避免依赖）。

Java 提供四种引用：强引用、软引用、弱引用、虚引用。强引用对象通常不会被 GC；软引用在内存紧张时回收；弱引用在下一次 GC 时即可回收；虚引用主要用于跟踪对象回收，不通过它取得对象本身。


## 九、String 与运行时常量池

字符串字面量会进入字符串常量池，池的实现位置随 JDK 版本变化（Java 7 起主要位于堆）。`String.intern()` 返回池中与内容相同的规范实例；它不等于“所有字符串都自动只保留一份”，`new String("a")` 仍会先创建对象，再根据调用方式决定是否复用池中实例。

面试回答应明确 `==` 比较引用身份，`equals` 比较内容；编译期可确定的字符串常量会被编译器折叠，运行期拼接则通常创建新的对象。


## 十、类加载生命周期

一个类从字节码到可使用，大致经历：

1. **加载**：通过类的全限定名找到二进制字节流，转化为方法区的运行时数据结构，并创建 `Class` 对象；
2. **链接**：验证字节码安全性，准备静态变量默认值，解析符号引用（可延迟）；
3. **初始化**：执行类变量显式赋值和静态代码块，按主动使用触发。

初始化通常由首次主动使用触发，例如 `new`、访问非编译期常量的静态字段、调用静态方法、反射加载等。被动使用编译期常量、仅声明数组类型通常不会触发目标类初始化。


## 十一、类加载器与双亲委派

类加载器负责把类名映射到字节码。常见加载器包括 Bootstrap ClassLoader、Platform/Extension ClassLoader、Application ClassLoader，以及自定义 ClassLoader。

双亲委派模型的流程是：当前加载器先把请求交给父加载器；父加载器无法完成时，子加载器再尝试加载。它能避免核心类被业务类替换，并保证类的层次一致性。需要打破委派时（如容器隔离、SPI、热部署）应谨慎设计自定义加载器和命名空间。

类的唯一身份由“类加载器 + 全限定类名”共同决定；同名类由不同加载器加载，彼此也可能不能强制转换。


## 十二、垃圾回收算法

- **标记-清除**：标记不可达对象后清除，速度较快但会产生内存碎片；
- **标记-复制**：把存活对象复制到另一块空间，适合存活率低的年轻代，但需要预留空间；
- **标记-整理**：标记后让存活对象向一端移动，再清理边界外空间，适合老年代但移动成本更高；
- **分代收集**：根据对象生命周期选择算法，年轻代偏向复制，老年代常用标记-整理或标记-清除的组合。

算法选择的核心目标是平衡吞吐量、停顿时间、内存占用和碎片。


## 十三、垃圾收集器与 GC 类型

常见收集器包括 Serial、ParNew、Parallel、CMS、G1，以及较新的 ZGC、Shenandoah。回答时不要只背名称，要说明目标：

- Serial：单线程收集，简单但停顿明显，适合小堆或单核场景；
- Parallel：强调吞吐量，适合批处理；
- CMS：以低停顿为目标，采用并发标记清除，可能产生碎片且有并发失败；
- G1：把堆划分为 Region，按收益优先回收，兼顾停顿目标与吞吐；
- ZGC/Shenandoah：面向超大堆和低停顿，具体可用性取决于 JDK 版本。

JDK 9 起 G1 成为默认收集器；CMS、偏向锁等历史实现已在新版本中逐步弃用或移除，生产环境应先确认目标 JDK。


### G1 的核心思路

G1 不再把整个堆固定切成连续的新生代/老年代，而是划分为多个 Region，并在逻辑上维护年轻代、老年代和大对象区域。Young GC 复制存活对象；Mixed GC 在回收年轻代的同时选择部分老年代 Region。

G1 通过 Remembered Set 记录跨 Region 引用，避免每次扫描整个堆。`-XX:MaxGCPauseMillis` 是停顿目标而非硬性保证；当实时存活对象太多或堆空间不足时，仍可能出现长停顿甚至 Full GC。


## 十四、Full GC、内存泄漏与 OOM

**内存泄漏**是对象仍被不必要的强引用持有，导致 GC 无法回收；**内存溢出**是申请内存时已没有足够可用空间。泄漏可能最终导致 OOM，但两者不是同义词。

常见泄漏来源：静态集合无限增长、缓存没有淘汰、监听器未注销、ThreadLocal 在线程池中只 set 不 remove、类加载器被长生命周期对象引用。OOM 的第一步不是盲目增大堆，而是保留现场、确认是哪一块内存耗尽，再分析对象增长和引用链。


## 十五、JVM 参数与 GC 日志

常用参数按目标分组：

- 堆：`-Xms`、`-Xmx`、`-Xmn`；
- 栈：`-Xss`；
- 元空间：`-XX:MetaspaceSize`、`-XX:MaxMetaspaceSize`；
- GC 选择：`-XX:+UseG1GC`、`-XX:+UseParallelGC` 等；
- 日志：JDK 9+ 推荐统一日志 `-Xlog:gc*`，JDK 8 常见 `-XX:+PrintGCDetails -XX:+PrintGCTimeStamps`；
- 堆转储：`-XX:+HeapDumpOnOutOfMemoryError -XX:HeapDumpPath=...`。

参数必须结合目标 JDK 验证；同一个参数在不同版本可能被弃用、改名或含义变化。


## 十六、线上问题排查流程

1. **确认症状**：响应变慢、频繁 GC、OOM、CPU 飙高还是线程数异常；
2. **保留证据**：应用日志、GC 日志、堆转储、线程栈、进程和容器资源指标；
3. **选择工具**：`jps` 看进程，`jstat -gcutil` 看 GC 概况，`jstack` 看线程栈，`jmap` 或 `jcmd` 导出堆和类信息；
4. **分析根因**：MAT/VisualVM 查看大对象、Dominator Tree 和引用链，确认是泄漏、流量突增、缓存配置还是参数不匹配；
5. **验证修复**：先在压测或灰度环境验证，再调整堆、收集器、队列和业务代码。

`jmap`、`jstat` 等工具可能受容器权限、JDK 版本和安全策略影响，不能只凭一条命令下结论。


## 十七、常见面试速答

- **堆和栈的区别？** 堆是线程共享、存对象实例、由 GC 管理；栈是线程私有、以栈帧保存方法执行状态，方法结束后栈帧弹出。
- **为什么要分代？** 大多数对象朝生夕死，按生命周期分区可减少扫描和复制成本。
- **Minor GC 会 Stop-The-World 吗？** 传统收集器通常会；并发收集器也可能有短暂停顿，不能绝对说“没有停顿”。
- **类加载器为什么要双亲委派？** 防止核心类被替换、避免重复加载，并保持类身份和安全边界。
- **如何判断对象可回收？** 从 GC Roots 做可达性分析；不可达对象只是具备回收资格，最终是否马上回收由 GC 决定。
- **如何处理 OOM？** 先区分堆、元空间、直接内存或本地线程问题，保留转储和日志，沿引用链定位根因后再改代码或参数。


## 十八、自测清单

1. JVM 为什么能实现跨平台？
2. 程序计数器为什么是线程私有的？
3. 一个 Java 方法调用会在虚拟机栈中创建什么？
4. `-Xss` 调大后为什么可能创建不了更多线程？
5. Java 7 永久代与 Java 8 元空间有什么区别？
6. TLAB 解决了什么并发分配问题？
7. GC Roots 包括哪些典型对象？
8. 四种引用分别适合什么场景？
9. 类加载的加载、链接、初始化如何区分？
10. 双亲委派能否被打破？为什么容器需要类隔离？
11. 标记-清除、复制、整理的优缺点是什么？
12. G1 的 Region、Remembered Set 和 Mixed GC 分别做什么？
13. 如何从 GC 日志判断是对象分配过快还是老年代回收失败？
14. ThreadLocal 为什么可能泄漏，线程池中应该怎么写？


## 资料说明与版本提示

- 课件在线预览标题为 **《09-JVM相关面试题.pptx》**，共 135 页；本篇依据课件中的标题、内存结构图、代码提示和总结页整理。
- 课件页面存在重复页、OCR 断行和示例代码排版问题，已按知识点合并；没有把演示代码当作完整可运行工程。
- 未观看“视频”目录中的课程视频，未运行“代码”目录或课程工程，未进行压测；参数、收集器和线程数仍需根据实际 JDK、业务指标与监控数据验证。
- JVM 实现细节随 JDK 版本变化，尤其是永久代/元空间、偏向锁、CMS、G1 默认值和统一 GC 日志格式。面试时先说明目标 JDK，再回答实现细节。
